# ENNI analysis workbook

**Working analysis file.** This notebook audits the source transcripts and is not assessed report text. The MLCU section runs only after the group has completed and checked the scoring sheet using the official guide.

In [ ]:
from __future__ import annotations

import csv
import random
from collections import Counter
from pathlib import Path
from statistics import mean

from IPython.display import SVG, display

PROJECT = Path.cwd()
if not (PROJECT / "data" / "processed" / "enni_inventory.csv").exists():
    PROJECT = Path.cwd().parent
INVENTORY_PATH = PROJECT / "data" / "processed" / "enni_inventory.csv"
SCORES_PATH = PROJECT / "data" / "processed" / "mlcu_scoring_template.csv"

def read_csv(path: Path) -> list[dict[str, str]]:
    with path.open(encoding="utf-8", newline="") as handle:
        return list(csv.DictReader(handle))

inventory = read_csv(INVENTORY_PATH)
print(f"Transcripts: {len(inventory)}")
print("Folder groups:", dict(Counter(row["group_folder"] for row in inventory)))
print("Age missing:", sum(not row["age_years"] for row in inventory))
print("Child turns:", sum(int(row["child_turns"]) for row in inventory))
for marker in ("bch_turns", "maze_turns", "filler_turns", "breakoff_turns"):
    print(f"{marker}:", sum(int(row[marker]) for row in inventory))
print("Repeated story markers:", sum(int(row["duplicate_story_markers"]) > 0 for row in inventory))

## Age and sample description

The first figure describes the age coverage in the corpus. The one participant without age is retained in the inventory but cannot enter the age association analysis.

In [ ]:
def svg_age_chart(rows: list[dict[str, str]]) -> str:
    counts = {group: Counter() for group in ("SLI", "TD")}
    for row in rows:
        if row["age_years"]:
            counts[row["group_folder"]][int(float(row["age_years"]))] += 1
    bins = list(range(min(age for group in counts for age in counts[group]), max(age for group in counts for age in counts[group]) + 1))
    width, height, left, bottom = 720, 400, 65, 330
    maximum = max(counts[group][age] for group in counts for age in bins)
    scale, step = 230 / maximum, 600 / len(bins)
    bar_width = step * 0.32
    parts = [f'<svg xmlns="http://www.w3.org/2000/svg" width="{width}" height="{height}" viewBox="0 0 {width} {height}">',
             '<rect width="100%" height="100%" fill="white"/><style>text{font:13px sans-serif;fill:#111}.axis{stroke:#444}.sli{fill:#c44}.td{fill:#4682b4}</style>',
             f'<text x="{width/2}" y="24" text-anchor="middle">Participant age (whole-year bins)</text>',
             f'<line class="axis" x1="{left}" y1="{bottom}" x2="690" y2="{bottom}"/>',
             f'<line class="axis" x1="{left}" y1="60" x2="{left}" y2="{bottom}"/>']
    for tick in range(0, maximum + 1, 10):
        y = bottom - tick * scale
        parts.append(f'<text x="{left-10}" y="{y+4:.1f}" text-anchor="end">{tick}</text>')
    for index, age in enumerate(bins):
        x = left + index * step + step * 0.15
        for group, offset, css in (("SLI", 0, "sli"), ("TD", bar_width * 1.15, "td")):
            value = counts[group][age]
            bar_height = value * scale
            parts.append(f'<rect class="{css}" x="{x+offset:.1f}" y="{bottom-bar_height:.1f}" width="{bar_width:.1f}" height="{bar_height:.1f}"/>')
        parts.append(f'<text x="{x+bar_width:.1f}" y="{bottom+22}" text-anchor="middle">{age}</text>')
    parts.extend(['<text x="580" y="54" fill="#c44">SLI</text>', '<text x="630" y="54" fill="#4682b4">TD</text>',
                  '<text x="370" y="380" text-anchor="middle">Age (years)</text>',
                  '<text x="18" y="200" transform="rotate(-90 18 200)" text-anchor="middle">Participants</text>', '</svg>'])
    return "".join(parts)

figures_dir = PROJECT / "report" / "figures"
figures_dir.mkdir(parents=True, exist_ok=True)
age_svg = svg_age_chart(inventory)
(figures_dir / "age_distribution.svg").write_text(age_svg, encoding="utf-8")
display(SVG(age_svg))

## Provisional MLCU outcome

The scoring rule below approximates the ENNI guide from the CHAT main tiers: child turns marked `[+ bch]` and explicit break-off markers are excluded, and common retracing/maze spans and fillers are removed from word counts. This automation does not identify every question response or unrelated utterance; interpret the resulting MLCU as provisional and inspect the original transcripts before relying on it.

In [ ]:
import sys
sys.path.insert(0, str(PROJECT / "scripts"))
from analyse_enni import score_transcript, slope, permutation_test

raw_dir = PROJECT / "data" / "raw" / "ENNI"
records = [score_transcript(path) for path in sorted(raw_dir.rglob("*.cha"))]
score_path = PROJECT / "data" / "processed" / "mlcu_provisional.csv"
with score_path.open("w", encoding="utf-8", newline="") as handle:
    writer = csv.DictWriter(handle, fieldnames=list(records[0]))
    writer.writeheader()
    writer.writerows(records)
scored = [row for row in records if row["age_years"] != "" and row["eligible_c_units"]]
age = [float(row["age_years"]) for row in scored]
mlcu = [float(row["mlcu_provisional"]) for row in scored]
observed_slope, permutation_p, null_slopes = permutation_test(age, mlcu)
print(f"Provisional analysis sample: n={len(scored)}")
print(f"Pooled slope: {observed_slope:.4f} MLCU words/unit per year")
print(f"Two-sided permutation p-value: {permutation_p:.4f} (10,000 shuffles; seed 20261001)")
question_sensitive = [score_transcript(path, exclude_question_responses=True) for path in sorted(raw_dir.rglob("*.cha"))]
question_sensitive = [row for row in question_sensitive if row["age_years"] != "" and row["eligible_c_units"]]
sensitivity_slope, sensitivity_p, _ = permutation_test(
    [float(row["age_years"]) for row in question_sensitive],
    [float(row["mlcu_provisional"]) for row in question_sensitive],
)
print(f"Question-response sensitivity slope: {sensitivity_slope:.4f}; p={sensitivity_p:.4f}")
print("Sensitivity excludes every child turn immediately after an examiner question; this broad heuristic may over-exclude.")
print("The analysis is conditional on the provisional automated scoring rules above.")

## Primary analysis: pooled age–MLCU slope

The null distribution is generated by shuffling participant MLCU values relative to age and recalculating the simple-regression slope 10,000 times. The test is two-sided and uses a plus-one p-value correction. This tests a zero pooled association; it does not establish that age causes a change in narrative complexity. The score exclusions are approximate and need review.

In [ ]:
def svg_scatter(rows: list[dict[str, object]], fitted_slope: float) -> str:
    width, height, left, top, bottom = 720, 430, 70, 45, 350
    x_min, x_max, y_min, y_max = 4.0, 10.0, 2.0, 11.0
    sx = lambda value: left + (value - x_min) / (x_max - x_min) * 600
    sy = lambda value: bottom - (value - y_min) / (y_max - y_min) * 285
    x = [float(row["age_years"]) for row in rows]
    y = [float(row["mlcu_provisional"]) for row in rows]
    intercept = sum(y) / len(y) - fitted_slope * sum(x) / len(x)
    parts = [f'<svg xmlns="http://www.w3.org/2000/svg" width="{width}" height="{height}">',
             '<rect width="100%" height="100%" fill="white"/><style>text{font:13px sans-serif;fill:#111}.axis{stroke:#333}</style>',
             f'<line class="axis" x1="{left}" y1="{bottom}" x2="680" y2="{bottom}"/>',
             f'<line class="axis" x1="{left}" y1="{top}" x2="{left}" y2="{bottom}"/>']
    for tick in (4, 6, 8, 10):
        parts.append(f'<text x="{sx(tick):.1f}" y="{bottom+20}" text-anchor="middle">{tick}</text>')
    for tick in (2, 4, 6, 8, 10):
        parts.append(f'<text x="{left-12}" y="{sy(tick)+4:.1f}" text-anchor="end">{tick}</text>')
    for row in rows:
        color = "#c44" if row["group"] == "SLI" else "#4682b4"
        parts.append(f'<circle cx="{sx(float(row["age_years"])):.1f}" cy="{sy(float(row["mlcu_provisional"])):.1f}" r="3" fill="{color}" fill-opacity=".55"/>')
    parts.append(f'<line x1="{sx(x_min):.1f}" y1="{sy(intercept+fitted_slope*x_min):.1f}" x2="{sx(x_max):.1f}" y2="{sy(intercept+fitted_slope*x_max):.1f}" stroke="#222" stroke-width="2"/>')
    parts.extend(['<text x="370" y="405" text-anchor="middle">Age (years)</text>',
                  '<text x="18" y="200" transform="rotate(-90 18 200)" text-anchor="middle">Provisional MLCU (words/C-unit)</text>',
                  '<text x="530" y="28" fill="#c44">SLI</text><text x="580" y="28" fill="#4682b4">TD</text></svg>'])
    return "".join(parts)

def svg_null_distribution(values: list[float], observed: float) -> str:
    width, height, left, bottom = 720, 380, 60, 320
    bins, low, high = 35, min(values), max(values)
    counts = [0] * bins
    for value in values:
        index = min(bins - 1, int((value - low) / (high - low) * bins))
        counts[index] += 1
    scale = 230 / max(counts)
    parts = [f'<svg xmlns="http://www.w3.org/2000/svg" width="{width}" height="{height}">',
             '<rect width="100%" height="100%" fill="white"/><style>text{font:13px sans-serif;fill:#111}.axis{stroke:#333}</style>',
             f'<line class="axis" x1="{left}" y1="{bottom}" x2="680" y2="{bottom}"/>']
    bar_width = 600 / bins
    for tick in (0, max(counts) // 2, max(counts)):
        y = bottom - tick * scale
        parts.append(f'<text x="{left-8}" y="{y+4:.1f}" text-anchor="end">{tick}</text>')
    for index, count in enumerate(counts):
        bar_height = count * scale
        parts.append(f'<rect x="{left+index*bar_width:.1f}" y="{bottom-bar_height:.1f}" width="{bar_width-1:.1f}" height="{bar_height:.1f}" fill="#8aa"/>')
    marker_x = left + (observed - low) / (high - low) * 600
    outside = marker_x < left or marker_x > 680
    marker_x = min(680, max(left, marker_x))
    label = "Observed slope is beyond the shuffled range" if outside else "Observed slope"
    parts.extend([f'<line x1="{marker_x:.1f}" y1="55" x2="{marker_x:.1f}" y2="{bottom}" stroke="#c33" stroke-width="2"/>',
                  f'<text x="370" y="35" text-anchor="middle" fill="#c33">{label}</text>',
                  '<text x="370" y="360" text-anchor="middle">Slope under shuffled age–MLCU pairings</text>',
                  '<text x="18" y="190" transform="rotate(-90 18 190)" text-anchor="middle">Shuffle count</text></svg>'])
    return "".join(parts)

scatter_svg = svg_scatter(scored, observed_slope)
null_svg = svg_null_distribution(null_slopes, observed_slope)
(figures_dir / "age_mlcu_provisional.svg").write_text(scatter_svg, encoding="utf-8")
(figures_dir / "permutation_null_provisional.svg").write_text(null_svg, encoding="utf-8")
display(SVG(scatter_svg))
display(SVG(null_svg))

## Secondary description: group-specific slopes

The group-specific slopes are descriptive estimates, not a formal test of whether the slopes differ. The pooled estimate can reflect both within-group age trends and differences between the SLI and TD groups.

In [ ]:
for group in ("SLI", "TD"):
    subset = [row for row in scored if row["group"] == group]
    estimate = slope([float(row["age_years"]) for row in subset], [float(row["mlcu_provisional"]) for row in subset])
    print(f"{group}: n={len(subset)}, descriptive slope={estimate:.4f} per year")

## Report handoff

Use `report/outline.md` for the group's own interpretation and limitations. Treat every score, exclusion rule, p-value, figure, and conclusion as provisional until the group has checked the scoring decisions against the transcripts. The assessment prohibits AI-generated content in the final submission; the group must write its own assessed prose.